### Configuração e Conexão com o Banco

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os

# configurações
DB_HOST = 'localhost' 
DB_PORT = '5432'
DB_NAME = 'ceap_dw'
DB_USER = 'admin'
DB_PASS = 'admin_password'

# Caminhos dos Arquivos
RAW_PATH_MAIN = '../data layer/raw/deputies_dataset.csv'
RAW_PATH_V2 = '../data layer/raw/dirty_deputies_v2.csv'

# Conexão Banco
db_url = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

print(" 1. Bibliotecas importadas e configuração de banco pronta.")

### EXTRACT (Extração)

In [ ]:
print(" Iniciando leitura dos arquivos RAW...")

try:
    df_raw = pd.read_csv(RAW_PATH_MAIN, low_memory=False)org
    print(f"   - Arquivo Principal carregado: {len(df_raw)} linhas.")

    df_v2 = pd.read_csv(RAW_PATH_V2, low_memory=False)
    print(f"   - Arquivo V2 (Enriquecimento) carregado: {len(df_v2)} linhas.")

except Exception as e:
    print(f" Erro na leitura dos arquivos: {e}")

### TRANSFORM (Limpeza e Padronização)

In [ ]:
print("Iniciando tratamento de dados...")

#  Filtra bugged_date
if 'bugged_date' in df_raw.columns:
    df_silver = df_raw[df_raw['bugged_date'] == 0].copy()
    df_silver = df_silver.drop(columns=['bugged_date'], errors='ignore')
else:
    df_silver = df_raw.copy()

#  Tipagem Numérica e Data
df_silver['receipt_value'] = pd.to_numeric(df_silver['receipt_value'], errors='coerce')
df_silver['receipt_date'] = pd.to_datetime(df_silver['receipt_date'], errors='coerce')

#  Padronização de Texto Geral
text_cols = ['deputy_name', 'political_party', 'establishment_name', 'receipt_description', 'receipt_social_security_number']
for col in text_cols:
    if col in df_silver.columns:
        df_silver[col] = df_silver[col].astype(str).str.upper().str.strip()


if 'deputy_state' in df_silver.columns:
    df_silver = df_silver.rename(columns={'deputy_state': 'state_code'})

if 'state_code' in df_silver.columns:
    df_silver['state_code'] = df_silver['state_code'].astype(str).str.upper().str.strip()
    
    #  Tratamento de "nan" (pandas transforma nulo em texto "nan")
    # Se for "NAN", vira string vazia ou None
    df_silver.loc[df_silver['state_code'] == 'NAN', 'state_code'] = None
    
    df_silver['state_code'] = df_silver['state_code'].str.slice(0, 2)
    
    print("   - Coluna 'state_code' tratada e truncada para 2 caracteres.")

print(f" 3. Limpeza concluída. Linhas prontas: {len(df_silver)}")

### ENRICH (Enriquecimento com Join)

In [ ]:

print(" Cruzando dados com a tabela V2 (Partidos)...")

if 'political_party' in df_v2.columns:
    # 4.1. Renomear ideologia 
    if 'party_ideology1' in df_v2.columns:
        df_v2 = df_v2.rename(columns={'party_ideology1': 'party_ideology'})

    # 4.2. Padroniza a chave
    df_v2['political_party'] = df_v2['political_party'].astype(str).str.upper().str.strip()
    

    if 'party_regdate' in df_v2.columns:
        df_v2['party_regdate'] = pd.to_datetime(df_v2['party_regdate'], dayfirst=True, errors='coerce')
        print("   - Datas de fundação dos partidos convertidas com sucesso.")

    # 4.4. Remove duplicatas na V2
    df_v2_unique = df_v2.drop_duplicates(subset=['political_party'])

    # 4.5. Faz o Join
    cols_v2 = ['political_party', 'party_regdate', 'party_ideology']
    cols_final_v2 = [c for c in cols_v2 if c in df_v2_unique.columns]
    
    # Atualiza o df_silver com as novas colunas
    df_silver = pd.merge(df_silver, df_v2_unique[cols_final_v2], on='political_party', how='left')
    
    print(" 4. Join realizado! Dados enriquecidos.")
else:
    print(" Aviso: Coluna 'political_party' não encontrada na V2.")

### LOAD (Carregar no Banco)

In [ ]:
#5. LOAD
from sqlalchemy import text

target_table = 'tb_reembolso'
target_schema = 'silver'

print(f"Inserindo dados na tabela '{target_schema}.{target_table}'...")

try:
    with engine.connect() as conn:
        # 1. Limpa os dados, mas MANTÉM a estrutura do DDL
        conn.execute(text(f"TRUNCATE TABLE {target_schema}.{target_table} RESTART IDENTITY;"))
        conn.commit()
        print("   - Tabela limpa (Truncate realizado).")

    # 2. Insere os dados novos (APPEND)
    df_silver.to_sql(
        target_table,
        engine,
        schema=target_schema,
        if_exists='append', 
        index=False
    )

    print(f" SUCESSO! Dados inseridos na tabela Silver.")

except Exception as e:
    print(f"Erro: {e}")